# 5. Shutter modes

Query mode and learn why selecting Manual is a separate, explicit operation.


## Before you start

Follow [the notebook setup and safety guide](README.md), then select the
project's Python 3.12 kernel. Run cells from top to bottom. Each notebook
is independent. Default execution uses a fictional config and makes no
hardware calls; a skipped operation is **not a successful hardware test**.

For an authorized live session, set `CONFIG` to `ROOT / 'shutters.local.toml'`,
set `TARGET` to the reviewed exact device name, and only then enable the
relevant gate. Keep the source independently protected. Release any GUI,
CLI or other notebook owner first. Restart the kernel after changing setup.


In [ ]:
from pathlib import Path
from dataclasses import asdict
from pprint import pprint
from thorlabs_shutter_control import ShutterHub, load_config

# Works from the repository root or docs/examples. No device access.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'config/shutters.example.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the project checkout.')
CONFIG = ROOT / 'config/shutters.example.toml'  # Fictional; offline only.
TARGET = 'example'  # Exact configured name; never select the first device.
RUN_HARDWARE = False

configs = load_config(CONFIG)
selected = next((c for c in configs if c.name == TARGET), None)
if selected is None:
    raise ValueError('TARGET must match an exact configured name.')
print('Selected:', selected.name, selected.serial_number, selected.route.kind)


## 1. Mode names and supported operations

| Driver readback | Meaning / current support |
| --- | --- |
| `manual` | Qualified software ON/OFF supported |
| `single` | Single-cycle mode reported; cycle configuration/start unavailable |
| `auto` | Automatic-cycle mode reported; cycle configuration/start unavailable |
| `triggered` | External-trigger mode reported; trigger setup/priming unavailable |

These are driver names, not a complete trigger-operation tutorial. Do not
infer TTL voltage, polarity, priming or safe disconnect behavior from a name.
See [protocol/source traceability](../INTERFACE.md).


In [ ]:
if RUN_HARDWARE:
    hub = ShutterHub(configs)
    try:
        hub.connect(TARGET)
        print('Current mode:', hub.get_status(TARGET).feedback.operating_mode)
    finally:
        hub.close()
else:
    print('Mode query skipped.')


## 2. Optional explicit Manual selection

KSC101 remembers a changed operating mode across power cycles (source
SRC-002 p20 in [the source register](../sources.md)). This is a persistent
change even without a separate EEPROM command. Keep the source inhibited,
establish OFF independently, preserve interlocks, and obtain authorization
covering the mode change before enabling the gate below.

A reviewed qualification policy is required. An actual transition also
needs `allow_persistent_mode_changes=true`. The driver requires electrical
OFF and configured interlock protection. It refuses unresolved external
trigger authority. Already Manual/OFF produces readback without rewriting.


In [ ]:
RUN_MODE_CHANGE = False
if RUN_HARDWARE and RUN_MODE_CHANGE:
    hub = ShutterHub(configs)
    try:
        hub.connect(TARGET)
        result = hub.set_manual_mode(TARGET)
        pprint(asdict(result))
    finally:
        hub.close()
else:
    print('Mode change skipped; no mode setting written.')


Expected live result: Manual with electrical OFF readback. Independently
verify the optical state; mode readback does not prove closure. Do not
automatically restore a prior cycle/trigger mode on exit. A refused mode
change is a boundary to investigate, not a reason to bypass the owner.
